In [2]:
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, 
    round as spark_round, max as spark_max,
    min as spark_min, current_timestamp, lit
)

# Always build Gold from Silver, never from Bronze
df_silver = spark.read.format("delta").table("silver_transactions")

print(f"Silver row count: {df_silver.count()}")
df_silver.show(3)

StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 4, Finished, Available, Finished, False)

Silver row count: 7808
+--------------+----------+-------------+---------+------+--------+---------+------+----+-----+-------+-----------+----------+-------------+--------------------+
|transaction_id|      date|     merchant| category|amount|currency|   status| layer|year|month|quarter|day_of_week|is_weekend|amount_bucket|    silver_timestamp|
+--------------+----------+-------------+---------+------+--------+---------+------+----+-----+-------+-----------+----------+-------------+--------------------+
|      TXN00001|2023-10-09|Electric Bill|Utilities| 13.42|     USD|completed|silver|2023|   10|      4|          2|     false|        Small|2026-03-03 03:03:...|
|      TXN00003|2023-04-06|         Aldi|Groceries|  16.1|     USD|completed|silver|2023|    4|      2|          5|     false|        Small|2026-03-03 03:03:...|
|      TXN00004|2024-07-28|    Uber Eats|   Dining| 242.2|     USD|completed|silver|2024|    7|      3|          1|      true|        Large|2026-03-03 03:03:...|
+----

In [3]:
# Monthly Spending

gold_monthly = df_silver \
    .groupBy("year", "month", "quarter") \
    .agg(
        spark_round(spark_sum("amount"), 2).alias("total_spent"),
        spark_round(avg("amount"), 2).alias("avg_transaction"),
        count("transaction_id").alias("transaction_count"),
        spark_round(spark_max("amount"), 2).alias("largest_transaction"),
        spark_round(spark_min("amount"), 2).alias("smallest_transaction")
    ) \
    .orderBy("year", "month")

gold_monthly.show(12)



StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 5, Finished, Available, Finished, False)

+----+-----+-------+-----------+---------------+-----------------+-------------------+--------------------+
|year|month|quarter|total_spent|avg_transaction|transaction_count|largest_transaction|smallest_transaction|
+----+-----+-------+-----------+---------------+-----------------+-------------------+--------------------+
|2023|    1|      1|    68343.5|         204.01|              335|             397.37|                4.28|
|2023|    2|      1|   60920.81|         211.53|              288|             399.61|                3.75|
|2023|    3|      1|   64195.14|         194.53|              330|             398.54|                 3.6|
|2023|    4|      2|   65057.47|         203.94|              319|              396.3|                4.01|
|2023|    5|      2|   67542.76|          205.3|              329|             399.63|                5.83|
|2023|    6|      2|   65562.59|         199.28|              329|             399.46|                3.83|
|2023|    7|      3|    6584

In [4]:
# Category Summary

gold_category = df_silver \
    .groupBy("category") \
    .agg(
        spark_round(spark_sum("amount"), 2).alias("total_spent"),
        spark_round(avg("amount"), 2).alias("avg_transaction"),
        count("transaction_id").alias("transaction_count")
    ) \
    .withColumn(
        "pct_of_total",
        spark_round(
            col("total_spent") / df_silver.agg(spark_sum("amount")).collect()[0][0] * 100,
            2
        )
    ) \
    .orderBy("total_spent", ascending=False)

gold_category.show()

StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 6, Finished, Available, Finished, False)

+-------------+-----------+---------------+-----------------+------------+
|     category|total_spent|avg_transaction|transaction_count|pct_of_total|
+-------------+-----------+---------------+-----------------+------------+
|Entertainment|  230354.31|         204.58|             1126|       14.49|
|    Transport|  228815.29|         207.45|             1103|       14.39|
|    Groceries|  228140.29|         209.69|             1088|       14.35|
|       Health|  227711.79|         206.07|             1105|       14.32|
|    Utilities|  221505.81|         198.84|             1114|       13.93|
|     Shopping|  219268.06|         202.09|             1085|       13.79|
|       Dining|  218185.42|         197.27|             1106|       13.72|
|Uncategorized|    16308.3|         201.34|               81|        1.03|
+-------------+-----------+---------------+-----------------+------------+



In [5]:
# Merchant Summary

gold_merchant = df_silver \
    .groupBy("merchant", "category") \
    .agg(
        spark_round(spark_sum("amount"), 2).alias("total_spent"),
        spark_round(avg("amount"), 2).alias("avg_transaction"),
        count("transaction_id").alias("visit_count")
    ) \
    .orderBy("total_spent", ascending=False)

# Top 20 merchants by spend
gold_merchant.show(20)

StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 7, Finished, Available, Finished, False)

+--------------+-------------+-----------+---------------+-----------+
|      merchant|     category|total_spent|avg_transaction|visit_count|
+--------------+-------------+-----------+---------------+-----------+
|       Netflix|Entertainment|   60790.52|          208.9|        291|
|  CVS Pharmacy|       Health|   59885.82|         211.61|        283|
|   Whole Foods|    Groceries|   59799.62|         215.11|        278|
| Electric Bill|    Utilities|   58671.22|         199.56|        294|
|  Trader Joe's|    Groceries|   58275.58|         215.84|        270|
|  AMC Theaters|Entertainment|   58274.96|         199.57|        292|
|          Aldi|    Groceries|   57698.62|         203.88|        283|
|    Phone Bill|    Utilities|   56918.33|         204.74|        278|
|       Spotify|Entertainment|   56666.03|         208.33|        272|
|       Dentist|       Health|   56405.72|         200.02|        282|
| Internet Bill|    Utilities|   56354.12|         195.67|        288|
|Gym M

In [10]:
# Daily Trends

gold_dailys = df_silver \
    .groupBy("date", "year", "month", "day_of_week", "is_weekend") \
    .agg(
        spark_round(spark_sum("amount"), 2).alias("daily_total"),
        count("transaction_id").alias("daily_transactions"),
        spark_round(avg("amount"), 2).alias("avg_transaction")
    ) \
    .withColumn("date", date_format(col("date"), "yyyy/MM/dd")) \
    .orderBy("date")

gold_daily.show(10)





StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 12, Finished, Available, Finished, False)

+----------+----+-----+-----------+----------+-----------+------------------+---------------+
|      date|year|month|day_of_week|is_weekend|daily_total|daily_transactions|avg_transaction|
+----------+----+-----+-----------+----------+-----------+------------------+---------------+
|2023/01/01|2023|    1|          1|      true|    2523.36|                14|         180.24|
|2023/01/02|2023|    1|          2|     false|    2129.67|                 7|         304.24|
|2023/01/03|2023|    1|          3|     false|    2398.06|                12|         199.84|
|2023/01/04|2023|    1|          4|     false|    2012.45|                10|         201.25|
|2023/01/05|2023|    1|          5|     false|    2360.38|                12|          196.7|
|2023/01/06|2023|    1|          6|     false|    1765.66|                11|         160.51|
|2023/01/07|2023|    1|          7|      true|    1580.83|                10|         158.08|
|2023/01/08|2023|    1|          1|      true|    2446.95|  

In [11]:
# write all 4 Gold Tables

# Helper function to write cleanly
def write_gold_table(df, table_name):
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)
    print(f"{table_name} written — {df.count()} rows")

write_gold_table(gold_monthly, "gold_monthly_spending")
write_gold_table(gold_category, "gold_category_summary")
write_gold_table(gold_merchant, "gold_merchant_summary")
write_gold_table(gold_dailys, "gold_daily_trends")

StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, 13, Finished, Available, Finished, False)

gold_monthly_spending written — 24 rows
gold_category_summary written — 8 rows
gold_merchant_summary written — 60 rows


AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'date' and 'date'

In [ ]:
# Quick sanity check across all gold tables
gold_tables = [
    "gold_monthly_spending",
    "gold_category_summary", 
    "gold_merchant_summary",
    "gold_daily_trends"
]

for table in gold_tables:
    df = spark.read.format("delta").table(table)
    print(f"\n{'='*40}")
    print(f" {table}")
    print(f"   Rows: {df.count()} | Columns: {len(df.columns)}")
    df.show(3)

StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

budget_data = [
    ("Groceries", 600.0),
    ("Dining", 400.0),
    ("Transport", 300.0),
    ("Entertainment", 200.0),
    ("Shopping", 500.0),
    ("Utilities", 250.0),
    ("Health", 150.0),
    ("Uncategorized", 100.0),
]

schema = StructType([
    StructField("category", StringType(), False),
    StructField("monthly_budget", DoubleType(), False),
])

df_budget = spark.createDataFrame(budget_data, schema)

write_gold_table(df_budget, "gold_budget_reference")
print("\n Budget reference table created")


StatementMeta(, 58e4f607-2601-43e2-8d57-b33114e39d2d, -1, Cancelled, , Cancelled, True)